In [ ]:
using Serialization
using Random
using DelimitedFiles

In [ ]:
# ============================================================
# 0. Load Ns and Ts
# ============================================================

# 
Ns_obj = deserialize("Ns.jls")
Ts_obj = deserialize("Ts.jls")

In [ ]:
# 
function extract_matrix(obj)
    if obj isa AbstractMatrix
        return obj
    elseif obj isa NamedTuple
        names = propertynames(obj)
        if :Ns_rat in names
            return obj.Ns_rat
        elseif :Ns in names
            return obj.Ns
        elseif :Ts in names
            return obj.Ts
        else
            error("NamedTuple  can not find Ns_rat / Ns / Ts")
        end
    else
        error("can not recognize: $(typeof(obj))")
    end
end

In [ ]:
Ns = extract_matrix(Ns_obj)   # should be 2304 × 151

In [ ]:
Ts = extract_matrix(Ts_obj)   # should be 2304 × 144, rank 141

In [ ]:
println("size(Ns) = ", size(Ns))

In [ ]:
println("size(Ts) = ", size(Ts))

In [ ]:
# ============================================================
# 1. Convert Rational / Float64 matrices to integer matrices
#    rank is unchanged after multiplying by a nonzero scalar.
# ============================================================

function integer_scale_matrix(M; float_tol=1e-12)
    if eltype(M) <: Rational
        den_lcm = BigInt(1)
        for x in M
            den_lcm = lcm(den_lcm, BigInt(denominator(x)))
        end

        A = Matrix{BigInt}(undef, size(M,1), size(M,2))
        for i in axes(M,1), j in axes(M,2)
            x = M[i,j]
            A[i,j] = BigInt(numerator(x)) * div(den_lcm, BigInt(denominator(x)))
        end

        return A, den_lcm

    elseif eltype(M) <: Integer
        return BigInt.(M), BigInt(1)

    elseif eltype(M) <: AbstractFloat
        R = Matrix{Rational{BigInt}}(undef, size(M,1), size(M,2))
        for i in axes(M,1), j in axes(M,2)
            R[i,j] = rationalize(BigInt, M[i,j]; tol=float_tol)
        end
        return integer_scale_matrix(R)

    else
        error("Unsupported element type: $(eltype(M))")
    end
end

In [ ]:
B, scale_Ns = integer_scale_matrix(Ns)

In [ ]:
C, scale_Ts = integer_scale_matrix(Ts)

In [ ]:
println("B = scale_Ns * Ns, scale_Ns = ", scale_Ns)

In [ ]:
println("C = scale_Ts * Ts, scale_Ts = ", scale_Ts)

In [ ]:
# ============================================================
# 2. Modular row rank tools
#    We use rank modulo a large prime.
#    For verification, use several primes.
# ============================================================

mutable struct ModBasis
    p::Int
    ncols::Int
    basis::Vector{Union{Nothing, Vector{Int}}}
    rank::Int
end

In [ ]:
function ModBasis(p::Int, ncols::Int)
    return ModBasis(p, ncols, Union{Nothing, Vector{Int}}[nothing for _ in 1:ncols], 0)
end

In [ ]:
function row_mod(row, p::Int, ncols::Int)
    v = Vector{Int}(undef, ncols)
    @inbounds for j in 1:ncols
        v[j] = Int(mod(row[j], p))
    end
    return v
end

In [ ]:
function reduce_with_basis!(v::Vector{Int}, st::ModBasis)
    p = st.p
    n = st.ncols

    @inbounds for j in 1:n
        if v[j] != 0 && st.basis[j] !== nothing
            brow = st.basis[j]::Vector{Int}
            coeff = v[j]
            for k in j:n
                v[k] = mod(v[k] - coeff * brow[k], p)
            end
        end
    end

    return v
end

In [ ]:
function is_independent(st::ModBasis, row)
    v = row_mod(row, st.p, st.ncols)
    reduce_with_basis!(v, st)

    @inbounds for x in v
        if x != 0
            return true
        end
    end
    return false
end

In [ ]:
function add_row!(st::ModBasis, row)
    p = st.p
    n = st.ncols

    v = row_mod(row, p, n)
    reduce_with_basis!(v, st)

    piv = 0
    @inbounds for j in 1:n
        if v[j] != 0
            piv = j
            break
        end
    end

    if piv == 0
        return false
    end

    invpiv = invmod(v[piv], p)

    @inbounds for j in piv:n
        v[j] = mod(v[j] * invpiv, p)
    end

    # Keep reduced echelon form:
    # eliminate the new pivot from old basis rows.
    @inbounds for j in 1:n
        if j != piv && st.basis[j] !== nothing
            brow = st.basis[j]::Vector{Int}
            coeff = brow[piv]
            if coeff != 0
                for k in piv:n
                    brow[k] = mod(brow[k] - coeff * v[k], p)
                end
            end
        end
    end

    st.basis[piv] = v
    st.rank += 1
    return true
end

In [ ]:
function rank_mod(A::AbstractMatrix{BigInt}, rows::Vector{Int}, p::Int)
    st = ModBasis(p, size(A,2))
    for i in rows
        add_row!(st, @view A[i,:])
    end
    return st.rank
end

In [ ]:
function rank_mod_allrows(A::AbstractMatrix{BigInt}, p::Int)
    return rank_mod(A, collect(1:size(A,1)), p)
end

In [ ]:
# ============================================================
# 3. Gap function
# ============================================================

const PSEARCH = 1000003

In [ ]:
const PVERIFY = [1000003, 1000033, 1000037]

In [ ]:
rNs_full = rank_mod_allrows(B, PSEARCH)

In [ ]:
rTs_full = rank_mod_allrows(C, PSEARCH)

In [ ]:
println("rank(Ns) mod p = ", rNs_full)

In [ ]:
println("rank(Ts) mod p = ", rTs_full)

In [ ]:
# Expected:
# rNs_full = 151
# rTs_full = 141

function gap_from_ranks(rNI, rGI, rNs_full, rTs_full)
    return (rNs_full - rNI) - (rTs_full - rGI)
end

In [ ]:
function verify_set(B, C, S; primes=PVERIFY)
    n = size(B,1)
    I = setdiff(1:n, S)

    records = []

    for p in primes
        rNfull = rank_mod_allrows(B, p)
        rGfull = rank_mod_allrows(C, p)

        rNI = rank_mod(B, I, p)
        rGI = rank_mod(C, I, p)

        dN = rNfull - rNI
        dG = rGfull - rGI
        gap = dN - dG

        push!(records, (p=p, rank_Ns_I=rNI, rank_Ts_I=rGI,
                        drop_Ns=dN, drop_Ts=dG, gap=gap))
    end

    return I, records
end

In [ ]:
# ============================================================
# 4. Greedy search
#
# Goal:
#   length(S) <= maxS
#   gap(S) > 0
#
# Since S = complement of I, this means length(I) >= 2304 - maxS.
# ============================================================

function greedy_positive_gap(B, C;
    maxS::Int=200,
    p::Int=PSEARCH,
    rng=Random.default_rng(),
    make_S_exact_maxS::Bool=false
)
    n = size(B,1)
    rNfull = rank_mod_allrows(B, p)
    rGfull = rank_mod_allrows(C, p)

    stN = ModBasis(p, size(B,2))
    stG = ModBasis(p, size(C,2))

    I = Int[]
    order = randperm(rng, n)

    # Initially I is empty, so gap = rNfull - rGfull = 10.
    # We add rows to I as long as positive gap remains.
    for row in order
        indepN = is_independent(stN, @view B[row,:]) ? 1 : 0
        indepG = is_independent(stG, @view C[row,:]) ? 1 : 0

        new_rN = stN.rank + indepN
        new_rG = stG.rank + indepG

        new_gap = gap_from_ranks(new_rN, new_rG, rNfull, rGfull)

        if new_gap > 0
            add_row!(stN, @view B[row,:])
            add_row!(stG, @view C[row,:])
            push!(I, row)
        end
    end

    sort!(I)
    S = setdiff(1:n, I)

    # Optional: enlarge S to exactly maxS if possible,
    # while keeping gap positive.
    if make_S_exact_maxS && length(S) < maxS
        rest = setdiff(1:n, S)
        shuffle!(rng, rest)

        for row in rest
            if length(S) >= maxS
                break
            end

            S_try = sort!(vcat(S, row))
            I_try = setdiff(1:n, S_try)

            rNI = rank_mod(B, I_try, p)
            rGI = rank_mod(C, I_try, p)
            g = gap_from_ranks(rNI, rGI, rNfull, rGfull)

            if g > 0
                S = S_try
            end
        end

        I = setdiff(1:n, S)
    end

    return sort(I), sort(S)
end

In [ ]:
# ============================================================
# 5. Search multiple sets
# ============================================================

function search_many_positive_gap(B, C;
    nsets::Int=20,
    maxS::Int=200,
    trials::Int=500,
    seed::Int=1234,
    make_S_exact_maxS::Bool=false,
    outdir::String="positive_gap_search_outputs"
)
    rng = MersenneTwister(seed)

    if !isdir(outdir)
        mkdir(outdir)
    end

    found = Dict{String, Tuple{Vector{Int}, Vector{Int}, Any}}()
    summary_rows = String[]

    push!(summary_rows,
        "id,#S,#I,rank_Ns_I,rank_Ts_I,151-rank_Ns_I,141-rank_Ts_I,gap"
    )

    for t in 1:trials
        I, S = greedy_positive_gap(
            B, C;
            maxS=maxS,
            rng=rng,
            make_S_exact_maxS=make_S_exact_maxS
        )

        if length(S) > maxS
            continue
        end

        Iverify, recs = verify_set(B, C, S)

        # require all primes give positive gap and same ranks
        gaps = [r.gap for r in recs]
        if !all(g -> g > 0, gaps)
            continue
        end

        # use sorted S as key
        key = join(S, ",")

        if haskey(found, key)
            continue
        end

        id = length(found) + 1
        found[key] = (I, S, recs)

        r0 = recs[1]

        writedlm(joinpath(outdir, "positive_gap_set_$(lpad(id,2,'0'))_S_1_based.txt"), S)
        writedlm(joinpath(outdir, "positive_gap_set_$(lpad(id,2,'0'))_I_1_based.txt"), I)

        push!(summary_rows,
            string(id, ",",
                   length(S), ",",
                   length(I), ",",
                   r0.rank_Ns_I, ",",
                   r0.rank_Ts_I, ",",
                   r0.drop_Ns, ",",
                   r0.drop_Ts, ",",
                   r0.gap)
        )

        println("found set ", id,
                ": #S=", length(S),
                ", #I=", length(I),
                ", drop_Ns=", r0.drop_Ns,
                ", drop_Ts=", r0.drop_Ts,
                ", gap=", r0.gap)

        if length(found) >= nsets
            break
        end
    end

    open(joinpath(outdir, "positive_gap_summary.csv"), "w") do io
        for line in summary_rows
            println(io, line)
        end
    end

    return found
end

In [ ]:
# ============================================================
# 6. Run
# ============================================================

# Case A:
# Search with #S <= 200, hence #I >= 2104.
found = search_many_positive_gap(
    B, C;
    nsets=10,
    maxS=400,
    trials=1000,
    seed=20260704,
    make_S_exact_maxS=false,
    outdir="positive_gap_S_le_400_date"
)

In [ ]:
# Case B:
# If you want exactly #S = 200, hence exactly #I = 2104,
# turn this on:
#
# found_exact = search_many_positive_gap(
#     B, C;
#     nsets=20,
#     maxS=200,
#     trials=1000,
#     seed=20260704,
#     make_S_exact_maxS=true,
#     outdir="positive_gap_S_eq_200"
# )